# Setup

In [1]:
import spacy
import tracery
import markovify
import random

In [2]:
# load a language model for English texts

nlp = spacy.load('en_core_web_md')

In [3]:
def unique(ls):
    """
    this function create a list of unique elements
    """
    unique = []
    for item in ls:
        if item not in unique:
            unique.append(item)
    return unique

In [4]:
def create_tag_dictionary(tokens):
    """
    this function creates a dictionary of POS tags; each tag contain words which belong to the category
    """
    
    tag_dictionary = {}
    
    for token in tokens:
        if token.tag_ not in tag_dictionary:
            ls = []
            tag_dictionary[token.tag_] = ls
            ls.append(token.text)
        else:
            tag_dictionary[token.tag_].append(token.text)

    return tag_dictionary

In [5]:
def tags_to_words(tag_sentence, dictionary):
    """
    this function turn tags into words
    the code below is generated by ChatGPT
    """
    
    output_words = []
    
    for tag in tag_sentence.split():
        if tag in dictionary and dictionary[tag]:
            word = random.choice(dictionary[tag])
            output_words.append(word)
        else:
            output_words.append(tag)  # fallback
        
    return " ".join(output_words)

# Ping Text Generator Using Spacy and Tracery

In [6]:
# create a create a Spacy object from Ping (1966) by Samuel Backett
# source: https://www.translatum.gr/forum/index.php?topic=176427.0

text = open("Ping.txt", "r", encoding="utf-8").read()
doc_1 = nlp(text)

In [7]:
# count tokens

# given the minimalism of words in this text, the stop words are retained

from collections import Counter

tokens = [token for token in doc_1 if token.is_alpha]

print("tokens:", len(tokens))

word_count = Counter([w.text for w in tokens])
print(word_count.most_common(20))

tokens: 934
[('white', 84), ('almost', 37), ('one', 30), ('only', 27), ('fixed', 25), ('light', 25), ('never', 23), ('blue', 22), ('just', 20), ('ping', 20), ('invisible', 15), ('known', 14), ('all', 14), ('front', 14), ('second', 14), ('eyes', 13), ('Ping', 13), ('yard', 12), ('the', 12), ('no', 11)]


In [8]:
# create list of words by different POS
# 1. tokenisation: extract words from the text and group them by their POS
# 2. lemmatisation: find the most basic form of each word
# 3. create a list of unique bases 

verbs = unique([token.lemma_ for token in doc_1 if token.pos_ == "VERB"])
print("Verbs:", verbs)

nouns = unique([token.lemma_ for token in doc_1 if token.pos_ == "NOUN"])
print("Nouns:", nouns)

adjectives = unique([token.lemma_ for token in doc_1 if token.pos_ == "ADJ"])
print("Adjectives:", adjectives)

adverbs = unique([token.lemma_ for token in doc_1 if token.pos_ == "ADV"])
print("Adverbs:", adverbs)

prepositions = unique([token.text for token in doc_1 if token.dep_ == "prep"]) # prepositions words does not change forms
print("Prepositions:", prepositions)

Verbs: ['know', 'fix', 'join', 'sew', 'see', 'blur', 'hang', 'heel', 'shin', 'mean', 'head', 'light', 'murmur', 'give', 'rise', 'murmurs', 'hole', 'invisible', 'fall', 'tear', 'blue', 'pe', 'silence', 'implore', 'flash', 'eye']
Nouns: ['body', 'yard', 'leg', 'heat', 'floor', 'square', 'wall', 'ceiling', 'eye', 'trace', 'grey', 'hand', 'palm', 'foot', 'angle', 'plane', 'ping', 'sign', 'silence', 'brief', 'blur', 'heel', 'uncover', 'toe', 'light', 'front', 'way', 'nature', 'memory', 'meeting', 'meaning', 'sound', 'time', 'heart', 'breath', 'colour', 'one', 'nose', 'ear', 'hole', 'mouth', 'image', 'wind', 'nail', 'hair', 'scar', 'white', 'flesh', 'head', 'half', 'lash', 'flash', 'second']
Adjectives: ['white', 'bare', 'light', 'front', 'right', 'invisible', 'blue', 'naught', 'black', 'murmur', 'second', 'same', 'much', 'colour', 'infinite', 'square', 'grey', 'long', 'old', 'sound', 'dim', 'closed', 'far', 'last', 'unlustrous']
Adverbs: ['never', 'only', 'just', 'almost', 'together', 'else

In [9]:
# Ping is a distinct literary text with "with no verb or with only non-finite verbs (the present and past participles),
# and the conversion of some adjectives into verbs." (Şahin, 2023) - https://dergipark.org.tr/en/pub/dtcfdergisi/issue/77909/1098935

# given this stylistic, the base verbs will be made into past participles by inflection

from lemminflect import getInflection

verbs_past_participle = [getInflection(v, tag='VBN')[0] for v in verbs]

print("Verbs Past Participle:", verbs_past_participle)

Verbs Past Participle: ['known', 'fixed', 'joined', 'sewn', 'seen', 'blurred', 'hung', 'heeled', 'shinned', 'meant', 'headed', 'lighted', 'murmured', 'given', 'risen', 'murmursed', 'holed', 'invisibled', 'fallen', 'torn', 'blued', 'ped', 'silenced', 'implored', 'flashed', 'eyed']


In [10]:
# create a text generator using Tracery

rules = {
    "origin": "#adjective# #noun# #verb_past_participle# #preposition# #noun#",
    "adjective": adjectives,
    "noun": nouns,
    "verb_past_participle": verbs_past_participle,
    "preposition": prepositions,
    "noun": nouns
}

grammar = tracery.Grammar(rules)
for i in range(10):
    print(grammar.flatten("#origin#"))

bare plane headed as flash
black hole silenced on floor
blue yard eyed with wall
dim nature holed as nose
murmur sign seen as white
invisible hole implored like floor
bare yard hung with time
same foot heeled in wind
colour foot torn by heel
dim front invisibled over trace


# Jabberwocky Poem Generator Using Spacy and Markovify

In [11]:
# create a create a Spacy object from Jabberwocky (1871) by Lewis Carroll
# source: https://www.poetryfoundation.org/poems/42916/jabberwocky

poem = open("Jabberwocky.txt", "r", encoding="utf-8").read()
doc_2 = nlp(poem)

In [12]:
# create a list of tokens without stop words, also considering different cases

from spacy.lang.en.stop_words import STOP_WORDS

all_words = [token for token in doc_2 if token.is_alpha]
tokens = [word for word in all_words if word.text.lower() not in STOP_WORDS]
print("all words:", len(all_words), "tokens:", len(tokens))

word_count = Counter([w.text for w in tokens])
print(word_count.most_common(20))

all words: 167 tokens: 89
[('Jabberwock', 3), ('Twas', 2), ('brillig', 2), ('slithy', 2), ('toves', 2), ('gyre', 2), ('gimble', 2), ('wabe', 2), ('mimsy', 2), ('borogoves', 2), ('mome', 2), ('raths', 2), ('outgrabe', 2), ('Beware', 2), ('vorpal', 2), ('stood', 2), ('thought', 2), ('went', 2), ('son', 1), ('jaws', 1)]


In [13]:
# print the poem in POS tags

words_in_tag = " ".join([token.tag_ for token in tokens])
print(words_in_tag)

NNP NNP NNP NNP NNS NN NN JJ NNS NNP NNS VBP VB NNP NN NNS VBP NN VBP VB NNP NN NNP NNP NNP VBD JJ NN NN JJ NN NN NN VBD VBD NNP NN VBD RB NN JJ NN VBD NNP NNS NN VBD VBG NNP NN VBD VBD JJ NN VBD NN NN VBD JJ NN VBD JJ NN NN VBD NNP VB NNS NNP NN UH JJ NN NNP NNP VBD NN NNP NNP NNP NNP NNS NN NN JJ NNS NNP NNS VBP


In [14]:
# create a new poem from the original text

# build a model

model = markovify.NewlineText(words_in_tag, state_size=1)

for i in range(5):
    tag_sequence = model.make_short_sentence(30, tries=100)
    sentence = tags_to_words(tag_sequence, create_tag_dictionary(tokens))
    print(sentence)

Jabberwock wabe took took mome arms outgrabe
mome Callay raths bite
Twas frumious Come shun jaws outgrabe
Jabberwock tulgey snicker claws outgrabe
Callay Jabberwock toves flame catch


In [15]:
# create a new poem by combining models

# Ping model

tokens_ping = [token for token in doc_1 if token.is_alpha]

words_in_tag_ping = " ".join([token.tag_ for token in tokens_ping])

model_ping = markovify.NewlineText(words_in_tag_ping, state_size=2)

# Jabberwocky model

all_words = [token for token in doc_2 if token.is_alpha]
tokens_jabberwocky = [word for word in all_words if word.text.lower() not in STOP_WORDS]

words_in_tag_jabberwocky = " ".join([token.tag_ for token in tokens_jabberwocky])

model_jabberwocky = markovify.NewlineText(words_in_tag_jabberwocky, state_size=2)

# combine both models

model_combined = markovify.combine([ model_ping, model_jabberwocky ], [ 1, 1 ])

# code by ChatGPT to fix error when the output is None
for i in range(5):
    for attempt in range(10):  # try up to 10 times
        tag_sequence = model_combined.make_short_sentence(40, tries=100)
        if tag_sequence:
            break
    if not tag_sequence:
        print("Failed to generate a sentence.")
        continue
    sentence = tags_to_words(tag_sequence, create_tag_dictionary(tokens_ping + tokens_jabberwocky))
    print(sentence)


ceiling brillig murmur almost over no white uncover over
that known bare known hair over
Ping Ping Ping Ping brillig eyes light
all fixed that just seam almost never white fixed gimble over
seam Callay with fixed blue feet one head ceiling day over
